# Data Cleaning -- Pembrolizumab vs. Chemotherapy in MSI-H
**This notebook prepares Flatiron Health CSV files for patients with MSI-H metastatic colorectal cancer receiving first-line pembrolizumab or chemotherapy. Each CSV is cleaned using the flatiron_cleaner package. The cleaned dataframes are then merged into a single dataset, which will be used for survival analysis.**

## Import packages

In [1]:
import numpy as np
import pandas as pd

from flatiron_cleaner import DataProcessorColorectal, merge_dataframes

## Import data

In [2]:
df = pd.read_csv('../outputs/pembro_chemo_index.csv')

In [3]:
df.head(3)

,PatientID,LineName,StartDate
0,FFFB7D51F655C,pembro,2023-10-04
1,FB62DC3D3C20F,pembro,2025-06-05
2,F15B1F9F90957,pembro,2018-07-31


In [4]:
df.shape

(26405, 3)

In [5]:
# remove time portion and leave just YYYY-MM-DD
df['StartDate'] = df['StartDate'].astype(str).str.split(' ').str[0]

In [6]:
ids = df.PatientID.to_list()

## Clean CSV files 

In [7]:
# Initialize class 
processor = DataProcessorColorectal()

### Process Enhanced_MetastaticCRC.csv

In [8]:
enhanced_df = processor.process_enhanced(file_path = '../data/Enhanced_MetastaticCRC.csv',
                                         patient_ids = ids,
                                         drop_stage = True,
                                         drop_dates = False)

2026-03-20 23:23:25,175 - INFO - Successfully read Enhanced_MetastaticCRC.csv file with shape: (46877, 5) and unique PatientIDs: 46877
2026-03-20 23:23:25,175 - INFO - Filtering for 26405 specific PatientIDs
2026-03-20 23:23:25,184 - INFO - Successfully filtered Enhanced_MetastaticCRC.csv file with shape: (26405, 5) and unique PatientIDs: 26405
2026-03-20 23:23:25,201 - INFO - Successfully processed Enhanced_MetastaticCRC.csv file with final shape: (26405, 7) and unique PatientIDs: 26405


In [9]:
enhanced_df.shape

(26405, 7)

In [10]:
enhanced_df = pd.merge(enhanced_df, df[['PatientID', 'StartDate']], on = 'PatientID', how = 'left')

In [11]:
enhanced_df.shape

(26405, 8)

In [12]:
enhanced_df['StartDate'] = pd.to_datetime(enhanced_df['StartDate'])

In [13]:
enhanced_df['days_met_to_treatment'] = (enhanced_df['StartDate'] - enhanced_df['MetDiagnosisDate']).dt.days
enhanced_df['days_met_to_treatment'] = np.where(enhanced_df['days_met_to_treatment'] < 0 , 0, enhanced_df['days_met_to_treatment'])
enhanced_df['days_to_treatment_before_30d'] = np.where(enhanced_df['days_met_to_treatment'] < 30, 1, 0)

In [14]:
enhanced_df['days_diagnosis_to_met'] = np.where(enhanced_df['days_diagnosis_to_met'] < 0 , 0, enhanced_df['days_diagnosis_to_met'])
enhanced_df['days_diagnosis_to_met'] = enhanced_df['days_diagnosis_to_met'].fillna(0)

In [15]:
enhanced_df.GroupStage_mod.value_counts(dropna = False)

GroupStage_mod
IV         17098
III         5431
II          2460
unknown      766
I            649
0              1
Name: count, dtype: int64

In [16]:
enhanced_df['GroupStage_mod'] = enhanced_df["GroupStage_mod"].map({
    '0': 0, 
    'I': 1,
    'II': 2,
    'III': 3,
    'IV': 4,
    'unknown': np.nan
})

enhanced_df['GroupStage_mod_na'] = np.where(enhanced_df['GroupStage_mod'].isna(), 1, 0)

# impute 4 since stage IV is most common
enhanced_df['GroupStage_mod'] = enhanced_df['GroupStage_mod'].fillna(4)

In [17]:
enhanced_df['met_diagnosis_year'] = pd.to_numeric(enhanced_df['met_diagnosis_year'])
enhanced_df['before_2020'] = np.where(enhanced_df['met_diagnosis_year'] < 2020, 1, 0)

In [18]:
enhanced_df = enhanced_df.drop(columns = ['DiagnosisDate', 
                                          'MetDiagnosisDate', 
                                          'StartDate'])

### Process Demographics.csv 

In [19]:
demographics_df = processor.process_demographics(file_path = '../data/Demographics.csv',
                                                 index_date_df = df,
                                                 index_date_column = 'StartDate')

2026-03-20 23:23:25,277 - INFO - Successfully read Demographics.csv file with shape: (46877, 6) and unique PatientIDs: 46877
2026-03-20 23:23:25,295 - WARNING - Found 1 ages outside valid range (18-120)
2026-03-20 23:23:25,309 - INFO - Successfully processed Demographics.csv file with final shape: (26405, 6) and unique PatientIDs: 26405


In [20]:
demographics_df = demographics_df.copy()

In [21]:
demographics_df = demographics_df.query('age >= 18 and age <= 120', engine = 'python')

In [22]:
demographics_df.Gender.value_counts(dropna = False)

Gender
M      14890
F      11510
NaN        4
Name: count, dtype: int64

In [23]:
# Impute missing with most common sex (male)
demographics_df['sex_male'] = np.where(demographics_df['Gender'] == 'F', 0, 1)

In [24]:
demographics_df = demographics_df.drop(columns = ['Gender'])

### Process Enhanced_MetCRCBiomarkers.csv

In [25]:
biomarkers_df = processor.process_biomarkers(file_path = '../data/Enhanced_MetCRCBiomarkers.csv',
                                             index_date_df = df, 
                                             index_date_column = 'StartDate',
                                             days_before = None, 
                                             days_after = 30)

2026-03-20 23:23:25,564 - INFO - Successfully read Enhanced_MetCRCBiomarkers.csv file with shape: (207027, 15) and unique PatientIDs: 42112
2026-03-20 23:23:25,655 - INFO - Successfully merged Enhanced_MetCRCBiomarkers.csv df with index_date_df resulting in shape: (129375, 16) and unique PatientIDs: 24887
2026-03-20 23:23:26,088 - INFO - Successfully processed Enhanced_MetCRCBiomarkers.csv file with final shape: (26405, 5) and unique PatientIDs: 26405


In [26]:
biomarkers_df['BRAF_status'] = biomarkers_df['BRAF_status'].fillna('unknown')
biomarkers_df['KRAS_status'] = biomarkers_df['KRAS_status'].fillna('unknown')
biomarkers_df['NRAS_status'] = biomarkers_df['NRAS_status'].fillna('unknown')
biomarkers_df['MMR/MSI_status'] = biomarkers_df['MMR/MSI_status'].fillna('unknown')

In [27]:
biomarkers_df['MMR/MSI_status'].value_counts()

MMR/MSI_status
negative    16261
unknown      8925
positive     1219
Name: count, dtype: int64

### Process Enhanced_CRC_HER2.csv

In [28]:
her2_df = processor.process_her2(file_path = '../data/Enhanced_CRC_HER2.csv',
                                 index_date_df = df, 
                                 index_date_column = 'StartDate',
                                 days_before = None, 
                                 days_after = 30)

2026-03-20 23:23:26,134 - INFO - Successfully read Enhanced_CRC_HER2.csv file with shape: (31958, 8) and unique PatientIDs: 18810
2026-03-20 23:23:26,152 - INFO - Successfully merged Enhanced_CRC_HER2.csv df with index_date_df resulting in shape: (20882, 9) and unique PatientIDs: 12123
2026-03-20 23:23:26,254 - INFO - Successfully processed Enhanced_CRC_HER2.csv file with final shape: (26405, 3) and unique PatientIDs: 26405


In [29]:
her2_df['HER2_status'] = her2_df['HER2_status'].fillna('unknown')

In [30]:
her2_df.HER2_status.value_counts()

HER2_status
unknown     18515
negative     7405
positive      485
Name: count, dtype: int64

### Process ECOG.csv

In [31]:
ecog_df = processor.process_ecog(file_path = '../data/ECOG.csv', 
                                 index_date_df = df,
                                 index_date_column = 'StartDate',
                                 days_before = 90,
                                 days_after = 0,
                                 days_before_further = 180)

2026-03-20 23:23:26,650 - INFO - Successfully read ECOG.csv file with shape: (1017535, 4) and unique PatientIDs: 36246
2026-03-20 23:23:26,905 - INFO - Successfully merged ECOG.csv df with index_date_df resulting in shape: (737663, 5) and unique PatientIDs: 22055
2026-03-20 23:23:27,180 - INFO - Successfully processed ECOG.csv file with final shape: (26405, 3) and unique PatientIDs: 26405


In [32]:
ecog_df.ecog_index.value_counts(dropna = False)

ecog_index
NaN    8705
0      8315
1      7225
2      1793
3       350
4        17
Name: count, dtype: int64

In [33]:
ecog_df['ecog_index'] = ecog_df['ecog_index'].astype('float64')
ecog_df['ecog_index_na'] = np.where(ecog_df['ecog_index'].isna(), 1, 0)

# impute 0 for missing ECOG since most common
ecog_df['ecog_index'] = ecog_df['ecog_index'].fillna(0)

In [34]:
ecog_df['ecog_newly_gte2'] = ecog_df['ecog_newly_gte2'].fillna(0)

### Process Vitals.csv

In [35]:
vitals_df = processor.process_vitals(file_path = '../data/Vitals.csv',
                                     index_date_df = df,
                                     index_date_column = 'StartDate',
                                     weight_days_before = 90,
                                     days_after = 0,
                                     vital_summary_lookback = 180, 
                                     abnormal_reading_threshold = 1)

2026-03-20 23:24:01,297 - INFO - Successfully read Vitals.csv file with shape: (17264775, 16) and unique PatientIDs: 46724
2026-03-20 23:24:11,526 - INFO - Successfully merged Vitals.csv df with index_date_df resulting in shape: (12035304, 17) and unique PatientIDs: 26404
2026-03-20 23:24:15,835 - INFO - Successfully processed Vitals.csv file with final shape: (26405, 8) and unique PatientIDs: 26405


### Process Lab.csv

In [36]:
labs_df = processor.process_labs(file_path = '../data/Lab.csv',
                                 index_date_df = df,
                                 index_date_column = 'StartDate',
                                 additional_loinc_mappings = {'cea': ['2039-6'], 'ldh': ['2532-0', '14804-9']},
                                 days_before = 90,
                                 days_after = 0,
                                 summary_lookback = 180)

2026-03-20 23:25:33,250 - INFO - Successfully read Lab.csv file with shape: (47135286, 17) and unique PatientIDs: 44960
2026-03-20 23:26:07,251 - INFO - Successfully merged Lab.csv df with index_date_df resulting in shape: (32867816, 18) and unique PatientIDs: 25973
2026-03-20 23:27:09,790 - INFO - Successfully processed Lab.csv file with final shape: (26405, 86) and unique PatientIDs: 26405


### Process MedicationAdministration.csv

In [37]:
medications_df = processor.process_medications(file_path = '../data/MedicationAdministration.csv',
                                               index_date_df = df,
                                               index_date_column = 'StartDate',
                                               days_before = 90,
                                               days_after = 0)

2026-03-20 23:27:18,562 - INFO - Successfully read MedicationAdministration.csv file with shape: (6618222, 11) and unique PatientIDs: 38657
2026-03-20 23:27:21,510 - INFO - Successfully merged MedicationAdministration.csv df with index_date_df resulting in shape: (4971612, 12) and unique PatientIDs: 25658
2026-03-20 23:27:21,765 - INFO - Successfully processed MedicationAdministration.csv file with final shape: (26405, 9) and unique PatientIDs: 26405


### Process Diagnosis.csv

In [38]:
diagnosis_df = processor.process_diagnosis(file_path = '../data/Diagnosis.csv',
                                           index_date_df = df,
                                           index_date_column = 'StartDate',
                                           days_before = None,
                                           days_after = 0)

2026-03-20 23:27:24,196 - INFO - Successfully read Diagnosis.csv file with shape: (3378971, 6) and unique PatientIDs: 46877
2026-03-20 23:27:25,091 - INFO - Successfully merged Diagnosis.csv df with index_date_df resulting in shape: (2065078, 7) and unique PatientIDs: 26405
2026-03-20 23:27:28,727 - INFO - Successfully processed Diagnosis.csv file with final shape: (26405, 40) and unique PatientIDs: 26405


### Process Enhanced_Mortality_V2.csv

In [39]:
mortality_df = processor.process_mortality(file_path = '../data/Enhanced_Mortality_V2.csv',
                                           index_date_df = df, 
                                           index_date_column = 'StartDate',
                                           visit_path = '../data/Visit.csv', 
                                           telemedicine_path = '../data/Telemedicine.csv', 
                                           biomarkers_path = '../data/Enhanced_MetCRCBiomarkers.csv',
                                           her2_path = '../data/Enhanced_CRC_HER2.csv',
                                           oral_path = '../data/Enhanced_MetCRC_Orals.csv', 
                                           progression_path = '../data/Enhanced_MetCRC_Progression.csv',
                                           drop_dates = False)

2026-03-20 23:27:28,783 - INFO - Successfully read Enhanced_Mortality_V2.csv file with shape: (31065, 2) and unique PatientIDs: 31065
2026-03-20 23:27:28,806 - INFO - Successfully merged Enhanced_Mortality_V2.csv df with index_date_df resulting in shape: (26405, 3) and unique PatientIDs: 26405
2026-03-20 23:27:31,177 - INFO - The following columns ['last_visit_date', 'last_biomarker_date', 'last_her2_date', 'last_oral_date', 'last_progression_date'] are used to calculate the last EHR date
2026-03-20 23:27:31,192 - INFO - Successfully processed Enhanced_Mortality_V2.csv file with final shape: (26405, 6) and unique PatientIDs: 26405. There are 0 out of 26405 patients with missing duration values


In [40]:
mortality_df = mortality_df[['PatientID', 'event', 'duration']]

In [41]:
mortality_df = mortality_df.query('duration >= 0')

### Process Insurance.csv

In [42]:
insurance_df = processor.process_insurance(file_path = '../data/Insurance.csv',
                                           index_date_df = df,
                                           index_date_column = 'StartDate',
                                           days_before = None,
                                           days_after = 0,
                                           missing_date_strategy = 'liberal')

2026-03-20 23:27:31,595 - INFO - Successfully read Insurance.csv file with shape: (287348, 14) and unique PatientIDs: 44318
2026-03-20 23:27:31,782 - INFO - Successfully merged Insurance.csv df with index_date_df resulting in shape: (172907, 15) and unique PatientIDs: 25045
2026-03-20 23:27:32,106 - INFO - Successfully processed Insurance.csv file with final shape: (26405, 5) and unique PatientIDs: 26405


### Process SocialDeterminantsOfHealth.csv 

In [43]:
ses_df = pd.read_csv('../data/SocialDeterminantsOfHealth.csv')

In [44]:
ses_df.head(3)

,PatientID,SESIndex2015_2019
0,F80487DEC2205,3
1,F07B841D33A54,1 - Lowest SES
2,FF87520824DBA,5 - Highest SES


In [45]:
ses_df.SESIndex2015_2019.value_counts(dropna = False)

SESIndex2015_2019
4                  9127
3                  8303
2                  8138
5 - Highest SES    8030
1 - Lowest SES     7959
NaN                4886
Name: count, dtype: int64

In [46]:
ses_df['ses_mod'] = ses_df['SESIndex2015_2019'].map({
    '1 - Lowest SES' : 1, 
    '2': 2, 
    '3': 3, 
    '4': 4, 
    '5 - Highest SES': 5
})

ses_df['ses_mod_na'] = np.where(ses_df['ses_mod'].isna(), 1, 0)

# impute 4 for missing SES
ses_df['ses_mod'] = ses_df['ses_mod'].fillna(4)

In [47]:
ses_df = ses_df[['PatientID', 'ses_mod', 'ses_mod_na']]

In [48]:
ses_df = ses_df[ses_df.PatientID.isin(df.PatientID)]

## Merge dataframes

In [49]:
pembro_chemo_features_df = merge_dataframes(enhanced_df,
                                            demographics_df,
                                            biomarkers_df,
                                            her2_df,
                                            ecog_df,
                                            vitals_df,
                                            labs_df,
                                            medications_df,
                                            diagnosis_df, 
                                            mortality_df, 
                                            insurance_df,
                                            ses_df,
                                            merge_type = 'inner')

2026-03-20 23:27:32,158 - INFO - Anticipated number of merges: 11
2026-03-20 23:27:32,158 - INFO - Anticipated number of columns in final dataframe presuming all columns are unique except for PatientID: 170
2026-03-20 23:27:32,162 - INFO - Dataset 1 shape: (26405, 9), unique PatientIDs: 26405
2026-03-20 23:27:32,165 - INFO - Dataset 2 shape: (26404, 6), unique PatientIDs: 26404
2026-03-20 23:27:32,167 - INFO - Dataset 3 shape: (26405, 5), unique PatientIDs: 26405
2026-03-20 23:27:32,170 - INFO - Dataset 4 shape: (26405, 3), unique PatientIDs: 26405
2026-03-20 23:27:32,172 - INFO - Dataset 5 shape: (26405, 4), unique PatientIDs: 26405
2026-03-20 23:27:32,175 - INFO - Dataset 6 shape: (26405, 8), unique PatientIDs: 26405
2026-03-20 23:27:32,177 - INFO - Dataset 7 shape: (26405, 86), unique PatientIDs: 26405
2026-03-20 23:27:32,179 - INFO - Dataset 8 shape: (26405, 9), unique PatientIDs: 26405
2026-03-20 23:27:32,180 - INFO - Dataset 9 shape: (26405, 40), unique PatientIDs: 26405
2026-03-

In [50]:
pembro_chemo_features_df.shape

(26088, 170)

In [51]:
pembro_chemo_features_df = pembro_chemo_features_df.query('`MMR/MSI_status` == "positive"')

In [52]:
pembro_chemo_features_df.shape

(1208, 170)

## Export dataframe

In [53]:
pembro_chemo_features_df.to_csv('../outputs/pembro_chemo_features_df.csv', index = False)

In [54]:
# Save dtypes
pembro_chemo_features_df.dtypes.apply(lambda x: x.name).to_csv('../outputs/pembro_chemo_features_dtypes.csv')